In [15]:
%load_ext autoreload
%autoreload 2

import plotly.express as px
import torch
import torch.optim as optim
import utils
from models.custom_net import Net
from models.resnet_fine_tuning import get_resnet_model
from torch import nn
from torch.utils.tensorboard import SummaryWriter
from torchmetrics import MetricCollection
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassConfusionMatrix,
    MulticlassF1Score,
    MulticlassPrecision,
    MulticlassRecall,
)
from torchvision.transforms import v2
from train import Trainer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Custom_net

In [16]:
custom_net_train_transform = v2.Compose(
    [
        v2.ToImage(),
        v2.Resize(size=140),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomCrop(size=(130, 130)),
        v2.ColorJitter(brightness=0.1, contrast=0.1),
        v2.ToDtype(torch.float32, scale=True),
    ]
)
custom_net_test_transform = v2.Compose(
    [
        v2.ToImage(),
        v2.Resize(size=140),
        v2.CenterCrop(size=(130, 130)),
        v2.ToDtype(torch.float32, scale=True),
    ]
)

data = utils.get_datasets_and_loaders(
    train_transform=custom_net_train_transform,
    test_transform=custom_net_test_transform,
)

In [17]:
image, label = next(iter(data.train_loader))
image = image[0:8]
image = image.permute(0, 2, 3, 1)
fig = px.imshow(image, facet_col=0, facet_col_spacing=0.01, height=300)
for i in range(image.shape[0]):
    fig.update_annotations(
        {"text": data.classes[label[i]]}, selector={"text": f"facet_col={i}"}
    )
    # fig.layout.annotations[i]["text"] = classes[label[i]]
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

In [ ]:
num_epochs = 30

model = Net()

new_params = []
old_params = []
for name, param in model.named_parameters():
    if not any(sub in name for sub in ["final_linear", "conv1x1_2"]):
        old_params.append(param)
    else:
        new_params.append(param)

loaders = {"train_loader": data.train_loader, "valid_loader": data.valid_loader}

optimizer = optim.Adam(
    [{"params": old_params, "lr": 1e-6}, {"params": new_params, "lr": 1e-4}],
    weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[1e-5, 1e-3], total_steps=len(data.train_loader) * num_epochs
)
opt = utils.OptimizationSuite(
    criterion=nn.CrossEntropyLoss(), optimizer=optimizer, scheduler=scheduler
)

log = utils.LoggingSuite(
    metrics=MetricCollection(
        {
            "Accuracy": MulticlassAccuracy(num_classes=6, average="micro"),
            "Precision": MulticlassPrecision(num_classes=6, average="macro"),
            "Recall": MulticlassRecall(num_classes=6, average="macro"),
            "F1": MulticlassF1Score(num_classes=6, average="macro"),
        }
    ),
    conf_matrix=MulticlassConfusionMatrix(num_classes=6),
    writer=SummaryWriter(),
)

conf = utils.TrainerConfig(checkpoint_path="checkpoints/custom_net/")

trainer = Trainer(model, data, opt, log, conf)
trainer.load_checkpoint(
    path="checkpoints/custom_net/best_model_checkpoint.pth", verbose=False
);

In [ ]:
trainer.fit(
    num_epochs=num_epochs,
    start_epoch=trainer.load_checkpoint(
        path="checkpoints/custom_net/best_model_checkpoint.pth"
    ),
)

### Resnet

In [95]:
image_net_mean = [0.485, 0.456, 0.406]
image_net_std = [0.229, 0.224, 0.225]

resnet_train_transform = v2.Compose(
    [
        v2.ToImage(),
        v2.Resize(size=(224, 224)),
        v2.RandomHorizontalFlip(p=0.5),
        v2.ColorJitter(brightness=0.1, contrast=0.1),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=image_net_mean, std=image_net_std),
    ]
)
resnet_test_transform = v2.Compose(
    [
        v2.ToImage(),
        v2.Resize(size=(224, 224)),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=image_net_mean, std=image_net_std),
    ]
)

data = utils.get_datasets_and_loaders(
    train_transform=resnet_train_transform,
    test_transform=resnet_test_transform,
)

In [102]:
num_epochs = 10

resnet = get_resnet_model(classes=data.classes)

learnable_params = []
for name, param in resnet.named_parameters():
    if name in ["fc.weight", "fc.bias"]:
        learnable_params.append(param)
    else:
        param.requires_grad = False

loaders = {"train_loader": data.train_loader, "valid_loader": data.valid_loader}

optimizer = optim.Adam(params=learnable_params, lr=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3, total_steps=len(data.train_loader) * num_epochs
)
opt = utils.OptimizationSuite(
    criterion=nn.CrossEntropyLoss(), optimizer=optimizer, scheduler=scheduler
)

log = utils.LoggingSuite(
    metrics=MetricCollection(
        {
            "Accuracy": MulticlassAccuracy(num_classes=6, average="micro"),
            "Precision": MulticlassPrecision(num_classes=6, average="macro"),
            "Recall": MulticlassRecall(num_classes=6, average="macro"),
            "F1": MulticlassF1Score(num_classes=6, average="macro"),
        }
    ),
    conf_matrix=MulticlassConfusionMatrix(num_classes=6),
    writer=SummaryWriter(),
)

conf = utils.TrainerConfig(checkpoint_path="checkpoints/resnet/")

trainer = Trainer(resnet, data, opt, log, conf)
# start_epoch = trainer.load_checkpoint(
#     path="checkpoints/resnet/best_model_checkpoint.pth", verbose=False
# )

In [ ]:
trainer.fit(num_epochs=num_epochs, start_epoch=0)

## Тестирование

In [34]:
print(
    f"Количество параметров custom_net = {sum(p.numel() for p in model.parameters())}\n"
    f"Количество параметров resnet = {sum(p.numel() for p in resnet.parameters())}"
)

Количество параметров custom_net = 375622
Количество параметров resnet = 11179590


## TODO

1. Оценить производительность модели
2. Импортировать ResNet50 и сделать fine-tuning
3. Добавить в Trainer evaluate
4. Выводить confusion matrxi, f1 score, пару картинок, на которых были ошибки